# Fine-tuning pretrained transformers on Hinglish sentiment (GPU)

**Part of:** *Hinglish sentiment classification - TF-IDF vs BiLSTM+attention vs CNN vs Transformer*

This notebook is the **transfer-learning** arm of the study. The local project (CPU only) trains
BiLSTM / CNN / a from-scratch Transformer on ~3k sentences. Here we fine-tune **pretrained** models
on the **same train/val/test split** so the numbers are directly comparable.

| Model | Pretraining | Why it is interesting |
|---|---|---|
| `bert-base-multilingual-cased` (mBERT) | Wikipedia, 104 languages | classic multilingual baseline |
| `xlm-roberta-base` | CommonCrawl, 100 languages | stronger multilingual model |
| `google/muril-base-cased` | Indian languages incl. transliterated text | built for Indian-language text |
| `l3cube-pune/hing-bert` | Roman-script Hindi-English (Hinglish) social media text | pretrained on exactly this kind of text |

**Before you run:** Runtime -> Change runtime type -> **T4 GPU**. Full run (4 models x LR grid x 3 seeds) takes
roughly 40-70 min on a T4; set `QUICK = True` in the config cell for a ~10 min first pass.

**Protocol (same as the local models):** hyper-parameter (learning rate) chosen on the validation set with seed 42,
best epoch chosen on validation macro-F1, test set touched once per run, best config repeated over 3 seeds.

In [ ]:
# 1) Setup
!pip -q install transformers scikit-learn pandas matplotlib
import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, otherwise this will be very slow.")

In [ ]:
# 2) Configuration
QUICK = False            # True = 2 models, 1 learning rate, 1 seed (fast sanity run)

MODELS = {
    "mBERT":      "bert-base-multilingual-cased",
    "XLM-R base": "xlm-roberta-base",
    "MuRIL":      "google/muril-base-cased",
    "HingBERT":   "l3cube-pune/hing-bert",
}
LR_GRID = [2e-5, 5e-5]   # chosen on validation macro-F1 using the first seed
SEEDS   = [42, 43, 44]
MAX_LEN, BATCH, EPOCHS, WARMUP_FRAC, WEIGHT_DECAY = 64, 32, 5, 0.1, 0.01

if QUICK:
    MODELS = {k: MODELS[k] for k in ("XLM-R base", "HingBERT")}
    LR_GRID, SEEDS = [3e-5], [42]
print(list(MODELS), LR_GRID, SEEDS)

In [ ]:
# 3) Data: download the IIITH Hi-En code-mixed corpus and rebuild the SAME split as the local project
import os, re, subprocess, unicodedata
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

if not os.path.exists("Sub-word-LSTM"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DrImpossible/Sub-word-LSTM.git"], check=True)

rows = []
for line in open("Sub-word-LSTM/Data/IIITH_Codemixed.txt", encoding="utf-8"):
    p = line.rstrip("\n").split("\t")
    if len(p) == 4:
        rows.append((int(p[0]), p[1], int(p[2])))
df = pd.DataFrame(rows, columns=["id", "text_raw", "label"])      # 0=negative 1=neutral 2=positive

train, rest = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=42)
val, test   = train_test_split(rest, test_size=0.50, stratify=rest["label"], random_state=42)
train, val, test = (d.reset_index(drop=True) for d in (train, val, test))

# Fingerprint of the split used in the local project (row count, sum of ids). If these differ,
# your split is not identical and the comparison with the local results is not apples-to-apples.
EXPECTED = {"train": (3103, 7557575), "val": (388, 943202), "test": (388, 955839)}
for name, d in (("train", train), ("val", val), ("test", test)):
    got = (len(d), int(d["id"].sum()))
    status = "OK (identical to local split)" if got == EXPECTED[name] else f"MISMATCH, expected {EXPECTED[name]}"
    print(f"{name:5s} {got}  {status}")

# Same normalisation as the local project ("full" mode)
_URL, _MENTION, _HASHTAG = re.compile(r"https?://\S+|www\.\S+"), re.compile(r"@\w+"), re.compile(r"#(\w+)")
_REPEAT, _SPACES = re.compile(r"(.)\1{2,}"), re.compile(r"\s+")
def clean_text(text):
    t = unicodedata.normalize("NFKC", str(text)).lower()
    t = _URL.sub(" urltok ", t); t = _MENTION.sub(" usertok ", t); t = _HASHTAG.sub(r" \1 ", t)
    t = _REPEAT.sub(r"\1\1", t)
    return _SPACES.sub(" ", t).strip()
for d in (train, val, test):
    d["text"] = d["text_raw"].map(clean_text)
print(train[["text", "label"]].head(3).to_string())

In [ ]:
# 4) Fine-tuning loop (plain PyTorch, no Trainer -> no version-specific arguments, easy to explain)
import copy, math, random, time
import torch, torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"          # mixed precision (fp16) only on GPU

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def metrics(y, p):
    f = f1_score(y, p, average=None, labels=[0, 1, 2], zero_division=0)
    return dict(accuracy=accuracy_score(y, p), macro_f1=f1_score(y, p, average="macro", zero_division=0),
                f1_negative=f[0], f1_neutral=f[1], f1_positive=f[2])

@torch.no_grad()
def predict(model, ids, mask, bs=128):
    model.eval()
    out = []
    for i in range(0, len(ids), bs):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            lg = model(input_ids=ids[i:i + bs].to(device), attention_mask=mask[i:i + bs].to(device)).logits
        out.append(lg.float().cpu())
    return torch.cat(out)

def fine_tune_core(model, data, lr, seed):
    """data = dict(tr=(ids, mask, y), va=(...), te=(...)) with tensors. Returns metrics + test predictions."""
    set_seed(seed)
    model.to(device)
    ids, mask, y = data["tr"]
    counts = np.bincount(y.numpy(), minlength=3)
    class_w = torch.tensor(len(y) / (3 * counts), dtype=torch.float32, device=device)   # imbalance handling
    loss_fn = nn.CrossEntropyLoss(weight=class_w)
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=lr)
    steps_per_epoch = math.ceil(len(y) / BATCH)
    total, warm = EPOCHS * steps_per_epoch, max(1, int(WARMUP_FRAC * EPOCHS * steps_per_epoch))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))  # linear warmup + decay
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    g = torch.Generator().manual_seed(seed)

    best_f1, best_epoch, best_state, t0 = -1.0, 0, None, time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(len(y), generator=g)
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            xb, mb, yb = ids[idx].to(device), mask[idx].to(device), y[idx].to(device)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
                logits = model(input_ids=xb, attention_mask=mb).logits
            loss = loss_fn(logits.float(), yb)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        v_pred = predict(model, data["va"][0], data["va"][1]).argmax(1).numpy()
        v_f1 = metrics(data["va"][2].numpy(), v_pred)["macro_f1"]
        if v_f1 > best_f1:
            best_f1, best_epoch = v_f1, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    t_pred = predict(model, data["te"][0], data["te"][1]).argmax(1).numpy()
    out = metrics(data["te"][2].numpy(), t_pred)
    out.update(val_macro_f1=best_f1, best_epoch=best_epoch, train_seconds=time.time() - t0,
               n_params=sum(p.numel() for p in model.parameters()), pred=t_pred)
    return out

def fine_tune(model_id, lr, seed):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(model_id)
    def enc(df):
        e = tok(list(df["text"]), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        return e["input_ids"], e["attention_mask"], torch.tensor(df["label"].values)
    data = dict(tr=enc(train), va=enc(val), te=enc(test))
    set_seed(seed)   # classifier head initialisation depends on the seed
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=3)
    res = fine_tune_core(model, data, lr, seed)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

In [ ]:
# 5) Run the experiment: LR chosen on validation (first seed), then repeat with more seeds
results, preds = [], {}
for name, model_id in MODELS.items():
    try:
        print(f"\n=== {name}  ({model_id}) ===")
        tuned = []
        for lr in LR_GRID:
            r = fine_tune(model_id, lr, SEEDS[0])
            tuned.append((r["val_macro_f1"], lr, r))
            print(f"  lr={lr:g}: val macro-F1={r['val_macro_f1']:.3f} (best epoch {r['best_epoch']}, {r['train_seconds']:.0f}s)")
        _, best_lr, first = max(tuned, key=lambda t: t[0])
        runs = [first] + [fine_tune(model_id, best_lr, s) for s in SEEDS[1:]]
        f1s = np.array([r["macro_f1"] for r in runs])
        preds[name] = runs[0]["pred"]
        results.append(dict(
            model=name, hf_id=model_id, best_lr=best_lr, n_params=runs[0]["n_params"],
            macro_f1_mean=f1s.mean(), macro_f1_std=f1s.std(), macro_f1_per_seed=[round(float(x), 4) for x in f1s],
            accuracy_mean=np.mean([r["accuracy"] for r in runs]),
            f1_negative_mean=np.mean([r["f1_negative"] for r in runs]),
            f1_neutral_mean=np.mean([r["f1_neutral"] for r in runs]),
            f1_positive_mean=np.mean([r["f1_positive"] for r in runs]),
            mean_train_seconds=np.mean([r["train_seconds"] for r in runs]), seeds=SEEDS))
        print(f"  -> TEST macro-F1 {f1s.mean():.3f} +/- {f1s.std():.3f}")
    except Exception as e:          # e.g. model id typo / no internet: skip, keep going
        print(f"  SKIPPED {name}: {type(e).__name__}: {e}")

res_df = pd.DataFrame(results).sort_values("macro_f1_mean", ascending=False)
res_df.to_csv("colab_results.csv", index=False)
for name, p in preds.items():
    test.assign(pred=p).to_csv(f"preds_{name.replace(' ', '_')}_s{SEEDS[0]}.csv", index=False)
res_df[["model", "best_lr", "macro_f1_mean", "macro_f1_std", "accuracy_mean", "f1_negative_mean", "f1_neutral_mean", "f1_positive_mean"]].round(3)

In [ ]:
# 6) Confusion matrix + per-class report for the best model (first seed)
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
best_name = res_df.iloc[0]["model"]
p, yt = preds[best_name], test["label"].values
print(best_name); print(classification_report(yt, p, target_names=["negative", "neutral", "positive"], digits=3))
cm = confusion_matrix(yt, p, labels=[0, 1, 2]); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(4.6, 4))
ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), ["neg", "neu", "pos"]); ax.set_yticks(range(3), ["neg", "neu", "pos"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(best_name)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cmn[i, j]:.2f}\n(n={cm[i, j]})", ha="center", va="center", fontsize=8,
                color="white" if cmn[i, j] > .5 else "black")
fig.tight_layout(); fig.savefig("cm_best_pretrained.png", dpi=160)

In [ ]:
# 7) Download the result files, then put them in the project's results/ folder
import glob
files_to_get = ["colab_results.csv", "cm_best_pretrained.png"] + sorted(glob.glob("preds_*_s*.csv"))
try:
    from google.colab import files
    for f in files_to_get:
        files.download(f)
except ImportError:
    print("Not on Colab. Files are in the working directory:", files_to_get)

## What to do with the results
1. Copy `colab_results.csv` and `preds_*.csv` into the project's `results/` folder and `cm_best_pretrained.png` into `figures/`.
2. Run `python src/make_report_assets.py` - it merges these numbers into the comparison table and figures used by the report.

**How to read the numbers honestly:** the test set has 388 sentences, so a difference of 1-2 F1 points is within noise.
Compare *mean +/- std over seeds*, not single runs. If a pretrained model beats the from-scratch models by a clear
margin, the explanation is transfer learning (representations learned from far more text than our ~3k sentences);
if it does not, say so and discuss why (e.g. romanised Hindi is poorly covered by multilingual vocabularies).